# 05 - The hackathon data (R)

**File:** `notebooks/05_hackathon_data_r.ipynb`

**What this does:** Lists what is in the hackathon data folder, then opens one
file from it.

**How to run it:** Open in JupyterLab, check the kernel in the top-right corner
says **R**, then choose *Run > Run All Cells*.

**Inputs:** the hackathon data folder (see the first cell).

**Outputs:** a listing and one table, printed.

## 1. Where the data is

The hackathon data lives in a shared folder rather than in this repository --
the files are far too big for git.

If your copy is somewhere else, change the path in the cell below. That is the
only line in this notebook you should need to touch.

In [ ]:
# Change this if your copy of the data is somewhere else.
data_dir <- file.path(path.expand("~"), "shared-public", "GliderRodeo")

if (dir.exists(data_dir)) {
  cat("Found the data at:", data_dir, "\n")
} else {
  cat("Not found:", data_dir, "\n\n")
  cat("Edit data_dir above to point at your copy of the hackathon data.\n")
}

## 2. What is in there?

Folders are glider deployments; the loose files at the top apply to all of them.

In [ ]:
if (dir.exists(data_dir)) {
  for (name in sort(list.files(data_dir))) {
    path <- file.path(data_dir, name)
    if (dir.exists(path)) {
      files <- list.files(path, recursive = TRUE, full.names = TRUE)
      size <- sum(file.info(files)$size, na.rm = TRUE) / 1e6
      cat(sprintf("  %-26s %3d files  %8.1f MB\n", name, length(files), size))
    } else {
      cat(sprintf("  %-26s %3s         %8.1f MB\n", name, "", file.info(path)$size / 1e6))
    }
  }
} else {
  cat("No data folder -- see the cell above.\n")
}

## 3. Opening one file

`PAM_Glider_Specs.csv` is small and a good starting point: one row per glider,
saying which hydrophone and platform each one carried. If it is missing, this
falls back to whatever CSV it can find.

In [ ]:
table <- NULL

if (dir.exists(data_dir)) {
  target <- file.path(data_dir, "PAM_Glider_Specs.csv")

  if (!file.exists(target)) {
    found <- list.files(data_dir, pattern = "\\.csv$", recursive = TRUE, full.names = TRUE)
    target <- if (length(found) > 0) found[1] else NA
  }

  if (is.na(target)) {
    cat("No CSV files found in", data_dir, "\n")
  } else {
    cat("Opening:", sub(paste0("^", data_dir, "/"), "", target), "\n")
    table <- read.csv(target)
    cat(nrow(table), "rows,", ncol(table), "columns\n")
  }
}

if (!is.null(table)) head(table)

## 4. A note on the big files

Some of these files are large -- the `*_science_timeseries.csv` and
`*_flight_timeseries_engineering.csv` ones run to hundreds of megabytes. Loading
one whole can use several times its size in memory and kill your kernel.

When you need one, read just the columns you want:

```r
read.csv(path)[, c("time", "depth", "temperature")]   # or use data.table::fread
```

The small files -- `*_GPS_timeseries.csv`, `*_fieldDefinitions.csv` (which explains
what each column means) -- are a good place to start.

## Done

That is the whole tour. Pick a deployment, open its `*_fieldDefinitions.csv` to
see what the columns mean, and start building -- see the
[main README](../README.md) for how to get your work back to GitHub.